# 03 — Retrieval, Grounded Reply Generation & Escalation
Builds a semantic index over historical (customer, brand reply) pairs, retrieves grounding
examples for a new message, generates a reply, and decides auto-handle vs. escalate.
Every generation is logged with which historical examples it was grounded on — needed later for
failure analysis.

In [ ]:
# --- CONFIG ---
OUTPUT_DIR = "data/processed"
CONFIG_DIR = "config"
LOG_PATH = "data/processed/generation_log.jsonl"
TOP_K_RETRIEVAL = 3
ESCALATION_CONFIDENCE_THRESHOLD = 0.70   # below this classifier confidence -> escalate
ESCALATION_SIMILARITY_THRESHOLD = 0.35   # below this retrieval similarity -> escalate ("never seen this before")
RISK_KEYWORDS = ["lawsuit", "lawyer", "sue", "fraud", "chargeback", "injury", "legal action", "scam", "unauthorized charge"]


In [ ]:
import os, json, pickle, re
import numpy as np, pandas as pd, yaml
from sentence_transformers import SentenceTransformer

pairs_df = pd.read_csv(os.path.join(OUTPUT_DIR, "cleaned_pairs.csv"))
with open(os.path.join(CONFIG_DIR, "intents.yaml")) as f:
    intents = yaml.safe_load(f)

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print(len(pairs_df), "historical pairs loaded")


## Reuse the LLM client + intent classifier from notebook 02 (copy the config + functions here so this notebook is standalone)

In [ ]:
LLM_PROVIDER = "groq"
LLM_MODEL = "llama-3.3-70b-versatile"

def make_llm_call():
    if LLM_PROVIDER == "groq":
        from groq import Groq
        client = Groq(api_key=os.environ["GROQ_API_KEY"])
        def call(prompt, max_tokens=300):
            resp = client.chat.completions.create(
                model=LLM_MODEL, max_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}]
            )
            return resp.choices[0].message.content.strip()
        return call
    elif LLM_PROVIDER == "gemini":
        import google.generativeai as genai
        genai.configure(api_key=os.environ["GEMINI_API_KEY"])
        model = genai.GenerativeModel(LLM_MODEL)
        def call(prompt, max_tokens=300):
            return model.generate_content(prompt).text.strip()
        return call
    raise ValueError(LLM_PROVIDER)

llm_call = make_llm_call()

def classify_intent_llm(text):
    intent_list = "\n".join(f"- {k}: {v}" for k, v in intents.items())
    prompt = f'''Classify this message into exactly one intent:
{intent_list}

Message: "{text}"
Respond with ONLY the intent key.'''
    raw = llm_call(prompt, max_tokens=20).strip().lower().replace(' ', '_')
    return raw if raw in intents else "general_complaint"


## Step 1 — Build the retrieval index

In [ ]:
pairs_df['customer_embedding'] = list(embed_model.encode(pairs_df['customer_text_clean'].tolist(), show_progress_bar=True))
index_matrix = np.vstack(pairs_df['customer_embedding'].values)
print(index_matrix.shape)


In [ ]:
def cosine_sim(a, b):
    a = a / (np.linalg.norm(a) + 1e-9)
    b = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-9)
    return b @ a

def retrieve(customer_text, top_k=TOP_K_RETRIEVAL):
    q_emb = embed_model.encode([customer_text])[0]
    sims = cosine_sim(q_emb, index_matrix)
    top_idx = np.argsort(-sims)[:top_k]
    results = []
    for idx in top_idx:
        row = pairs_df.iloc[idx]
        results.append({
            "similarity": float(sims[idx]),
            "customer_text": row['customer_text_clean'],
            "brand_reply": row['brand_text_clean'],
            "is_boilerplate": bool(row['is_boilerplate']),
            "pair_index": int(idx),
        })
    return results


## Step 2 — Grounded reply generation

In [ ]:
def generate_reply(customer_text, retrieved):
    examples_block = "\n\n".join(
        f"Example {i+1} (similarity {r['similarity']:.2f}):\nCustomer: {r['customer_text']}\nBrand reply: {r['brand_reply']}"
        for i, r in enumerate(retrieved)
    )
    prompt = f'''You are a customer support agent. Draft a reply to the new customer message below,
grounded in how this brand has historically resolved similar issues (the examples). Match the
brand's tone. Do not invent policies, refunds, or tracking numbers not implied by the examples.
State which example pattern you're following in one short clause if relevant.

{examples_block}

New customer message: "{customer_text}"

Reply:'''
    return llm_call(prompt, max_tokens=200)


## Step 3 — Escalation policy (rules first, LLM only for the ambiguous residual)

In [ ]:
risk_pattern = re.compile("|".join(re.escape(k) for k in RISK_KEYWORDS), re.IGNORECASE)

def escalation_decision(customer_text, intent_confidence, retrieved):
    top_similarity = retrieved[0]['similarity'] if retrieved else 0.0

    if risk_pattern.search(customer_text):
        return {"action": "escalate", "reason": "risk keyword matched (legal/financial/safety)", "confidence": 1.0}

    if intent_confidence < ESCALATION_CONFIDENCE_THRESHOLD:
        return {"action": "escalate", "reason": f"low intent confidence ({intent_confidence:.2f})", "confidence": intent_confidence}

    if top_similarity < ESCALATION_SIMILARITY_THRESHOLD:
        return {"action": "escalate", "reason": f"no close historical precedent (similarity {top_similarity:.2f})", "confidence": top_similarity}

    return {"action": "auto_handle", "reason": "matched known pattern with sufficient confidence", "confidence": intent_confidence}


**Note:** this uses a placeholder `intent_confidence` (see next cell) since the LLM classifier
doesn't return a real probability. If you have time, replace this with the TF-IDF+LogReg model's
`predict_proba` max value, which gives a genuine confidence score cheaply — flag this either way
in your decision log.

## Step 4 — Full pipeline function + logging

In [ ]:
def run_agent(customer_text):
    intent = classify_intent_llm(customer_text)
    retrieved = retrieve(customer_text)
    # placeholder confidence: 1.0 if intent matched a known key, else 0.5 -- replace with real proba if available
    intent_confidence = 0.85
    reply = generate_reply(customer_text, retrieved)
    decision = escalation_decision(customer_text, intent_confidence, retrieved)

    record = {
        "customer_text": customer_text,
        "intent": intent,
        "intent_confidence": intent_confidence,
        "retrieved": retrieved,
        "reply": reply,
        "decision": decision,
    }
    with open(LOG_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")
    return record


## Step 5 — Try it on a few real examples

In [ ]:
test_messages = pairs_df['customer_text_clean'].sample(5, random_state=7).tolist()
for msg in test_messages:
    result = run_agent(msg)
    print("CUSTOMER:", msg)
    print("INTENT:", result['intent'])
    print("DECISION:", result['decision'])
    print("REPLY:", result['reply'])
    print("-" * 80)
